# TDBRAIN MDD — 치료 반응(Responder) 예측 초기 탐색

본 실험(`tdbrain_mdd_vs_hc_cbramod.ipynb`) 이전에 진행한 **탐색 실험**입니다. 본 실험과 설정이 다르므로 결과를 직접 비교하지 마세요.

| 항목 | 이 노트북 (탐색) | 본 실험 |
|---|---|---|
| 과제 | MDD 환자 안에서 Responder 예측 | MDD vs HC |
| CBraMod | braindecode 버전 (`braindecode/cbramod-pretrained`) | 원본 repo + 원본 가중치 |
| 입력 | EC, 30초 창, 표준편차 기준 창 제외 + clipping | EO·EC, 5초 구간, 잡음 처리 없음 |
| 특징 | 채널·시간 평균 200차원 → PCA 16 | 채널 유지 3,800차원 |

**내용:** Clinical-only / EEG-only / 결합(early·late·stacking) 비교, 라벨 순열 검정, 무작위 가중치 CBraMod 대조(눈 뜸 vs 감음 구분, Responder 예측)

> **데이터는 저장소에 포함되지 않습니다.** zip 비밀번호는 실행할 때 입력합니다.

## 0. 설치 + CBraMod (braindecode) 불러오기

In [ ]:
!pip install -q braindecode mne huggingface_hub openpyxl
!apt -qq install -y aria2 p7zip-full > /dev/null 2>&1

In [ ]:
import torch
from braindecode.models import CBraMod

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CBraMod.from_pretrained(
    'braindecode/cbramod-pretrained',
    return_encoder_output=True   # 분류 결과 대신 임베딩을 받음
)
model = model.to(device).eval()

In [ ]:
x = torch.randn(2, 19, 6000).to(device)   # (사람 수, 채널 19, 30초×200Hz)
with torch.no_grad():
    out = model(x)
print(out.shape)

## 1. 참가자 엑셀 올리기 (`TDBRAIN_participants_V3.1.xlsx`)

In [ ]:
import os
if not os.path.exists('/content/TDBRAIN_participants_V3.1.xlsx'):
    from google.colab import files
    files.upload()
print(os.path.exists('/content/TDBRAIN_participants_V3.1.xlsx'))

## 2. 데이터 받기 + MDD 피험자만 압축 해제
남은 용량이 40 GB 이상인지 먼저 확인하세요. 비밀번호는 TDBRAIN 다운로드 안내에 있는 zip 비밀번호를 입력합니다.

In [ ]:
!df -h /content          # 남은 용량 확인 (40GB 이상이면 충분)

In [ ]:
import subprocess
URL = 'https://brainclinics.com/images/downloads/TDBRAIN_Dataset_V3_1_Encr.zip'
for i in range(10):                           # 끊기면 이어받기
    !aria2c -x 8 -s 8 -c --max-tries=0 --retry-wait=5 --console-log-level=warn -d /content -o tdbrain.zip "$URL"
    if subprocess.run(['unzip', '-l', '/content/tdbrain.zip'], capture_output=True).returncode == 0:
        break
!ls -l /content/tdbrain.zip

In [ ]:
import pandas as pd
df = pd.read_excel('/content/TDBRAIN_participants_V3.1.xlsx')
mdd_ids = sorted(df[df.formal_status == 'MDD'].TDBRAIN_ID.unique())
print(len(mdd_ids), mdd_ids[:3])   # 169, ['sub-...', ...]

patterns = [f"TDBRAIN_Dataset_V3_1/{s}/*" for s in mdd_ids]
with open('/content/mdd_patterns.txt', 'w') as f:
    f.write("\n".join(patterns))

In [ ]:
import getpass
PW = getpass.getpass('zip password: ')
!rm -rf /content/tdbrain_mdd
r = subprocess.run(["7z", "x", f"-p{PW}", "/content/tdbrain.zip",
                    "-o/content/tdbrain_mdd", "-ir@/content/mdd_patterns.txt", "-y"],
                   capture_output=True, text=True)
del PW
print(r.stdout[-800:]); print(r.stderr[-500:])
!ls /content/tdbrain_mdd/TDBRAIN_Dataset_V3_1 | wc -l
!du -sh /content/tdbrain_mdd

Drive에 백업 (다음부터는 이 단계를 건너뛰고 Drive 경로를 바로 사용)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/tdbrain_mdd /content/drive/MyDrive/
!du -sh /content/drive/MyDrive/tdbrain_mdd

In [ ]:
!rm /content/tdbrain.zip

## 3. 경로와 채널 확인

In [ ]:
from pathlib import Path

EEG_ROOT = Path('/content/drive/MyDrive/tdbrain_mdd/TDBRAIN_Dataset_V3_1')
PARTICIPANTS_XLSX = Path('/content/TDBRAIN_participants_V3.1.xlsx')
CONDITION = 'restEC'

files = sorted(EEG_ROOT.glob(f'sub-*/ses-*/eeg/*task-{CONDITION}_eeg.bdf'))
print(len(files), 'files found')

In [ ]:
import mne
mne.set_log_level('WARNING')

raw = mne.io.read_raw_bdf(files[0], preload=False)
print('sfreq:', raw.info['sfreq'])
print('channels:', raw.ch_names)

STD19 = ['Fp1','Fp2','F7','F3','Fz','F4','F8','T7','C3','Cz','C4','T8','P7','P3','Pz','P4','P8','O1','O2']
ALIAS = {'T7':'T3','T8':'T4','P7':'T5','P8':'T6'}

def match_channels(ch_names):
    lower = {c.lower(): c for c in ch_names}
    picked, missing = [], []
    for ch in STD19:
        cand = [ch, ALIAS.get(ch, ch)]
        hit = next((lower[c.lower()] for c in cand if c.lower() in lower), None)
        (picked if hit else missing).append(hit or ch)
    return picked, missing

picked, missing = match_channels(raw.ch_names)
print('picked :', picked)
print('missing:', missing)

## 4. 전처리와 임베딩 함수

In [ ]:
import torch

def embed(x):
    x = torch.as_tensor(x, dtype=torch.float32, device=device)
    with torch.no_grad():
        out = model(x)              # (B, 19, 30, 200)
    out = out.mean(dim=(1, 2))      # 채널·시간 평균 → (B, 200)
    return out.cpu().numpy()

In [ ]:
import numpy as np

SFREQ, WIN_SEC = 200, 30

def load_windows(path):
    raw = mne.io.read_raw_bdf(path, preload=True)
    picked, missing = match_channels(raw.ch_names)
    if missing:
        raise ValueError(f'missing channels {missing}')
    raw.pick(picked)
    raw.filter(0.3, 75.0)
    raw.notch_filter(50.0)
    raw.resample(SFREQ)
    data = raw.get_data() * 1e6 / 100.0      # V → µV, 100으로 나눔
    win = SFREQ * WIN_SEC
    n = data.shape[1] // win
    return np.stack([data[:, i*win:(i+1)*win] for i in range(n)])

w = load_windows(files[0])
print('windows:', w.shape)                    # (4, 19, 6000) 예상
print('value range:', w.min().round(2), w.max().round(2))

e = embed(w).mean(axis=0)
print('embedding:', e.shape)                  # (200,) 예상

In [ ]:
print('std per window:', w.std(axis=(1, 2)).round(3))
print('abs percentiles (50/95/99):', np.percentile(np.abs(w), [50, 95, 99]).round(3))
print('channel std:', dict(zip(picked, w.std(axis=(0, 2)).round(2))))

## 5. 잡음 창 제외 기준
창별 표준편차를 보고 `STD_THR`(기본 1.0, 즉 100 μV 수준)보다 큰 창을 뺍니다. 깨끗한 창이 없으면 가장 덜 시끄러운 창 1개를 씁니다.

In [ ]:
stats = []
for f in files[:20]:
    w_ = load_windows(f)
    s = w_.std(axis=(1, 2))
    stats.append((f.name[:12], s.round(2), int((s < 1.0).sum())))
for name, s, n_clean in stats:
    print(name, s, '깨끗한 창:', n_clean, '/', len(s))

In [ ]:
STD_THR = 1.0

def subject_embedding(path):
    w = load_windows(path)                     # (4, 19, 6000)
    s = w.std(axis=(1, 2))
    keep = s < STD_THR
    if keep.sum() == 0:                        # 깨끗한 창이 하나도 없으면
        keep = s == s.min()                    # 가장 덜 시끄러운 창 1개 사용
    w = np.clip(w[keep], -5, 5)
    return embed(w).mean(axis=0), int(keep.sum())

e, n_used = subject_embedding(files[0])
print(e.shape, '사용한 창:', n_used)

## 6. Responder 정보가 있는 MDD 환자 임베딩 추출 (EC)

In [ ]:
import pandas as pd

df = pd.read_excel(PARTICIPANTS_XLSX)
mdd = (df[df.formal_status == 'MDD']
       .sort_values('sessID').groupby('TDBRAIN_ID').first().reset_index())
mdd = mdd[mdd.Responder.notna()].reset_index(drop=True)

emb, ok_ids, n_win, fails = [], [], [], []
for _, r in mdd.iterrows():
    sid, ses = r.TDBRAIN_ID, int(r.sessID)
    cand = list(EEG_ROOT.glob(f'{sid}/ses-{ses}/eeg/*task-{CONDITION}_eeg.bdf'))
    if not cand:
        fails.append((sid, 'no file')); continue
    try:
        e, n = subject_embedding(cand[0])
        emb.append(e); ok_ids.append(sid); n_win.append(n)
    except Exception as ex:
        fails.append((sid, str(ex)))

emb = np.stack(emb)
np.save('/content/mdd_cbramod_emb.npy', emb)
pd.DataFrame({'id': ok_ids, 'n_win': n_win}).to_csv('/content/mdd_cbramod_ids.csv', index=False)
print('embeddings:', emb.shape, '| failed:', len(fails))
print('사용한 창 수 분포:', pd.Series(n_win).value_counts().sort_index().to_dict())

## 7. Clinical-only / EEG-only / 결합 비교 (분할 seed 10개)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer

CLIN = ['age','gender','education','BDI_pre','sleep','coffee','cigarette','beer','meal','well','rTMS PROTOCOL']

d = mdd.set_index('TDBRAIN_ID').loc[ok_ids]
y = d.Responder.astype(int).values
Xc = d[CLIN].apply(pd.to_numeric, errors='coerce').values
Xe = emb
Xf = np.hstack([Xc, Xe])
nc, n_pca = Xc.shape[1], 16

clin_pipe = lambda: make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(C=0.1, max_iter=5000))
eeg_pipe  = lambda: make_pipeline(StandardScaler(), PCA(n_pca), LogisticRegression(C=0.1, max_iter=5000))
fuse_pipe = lambda: make_pipeline(
    ColumnTransformer([
        ('clin', make_pipeline(SimpleImputer(), StandardScaler()), list(range(nc))),
        ('eeg',  make_pipeline(StandardScaler(), PCA(n_pca)), list(range(nc, Xf.shape[1]))),
    ]),
    LogisticRegression(C=0.1, max_iter=5000))

def cv_auc(make, X, seeds=10):
    s = []
    for seed in range(seeds):
        cv = StratifiedKFold(5, shuffle=True, random_state=seed)
        p = cross_val_predict(make(), X, y, cv=cv, method='predict_proba')[:, 1]
        s.append(roc_auc_score(y, p))
    return np.mean(s), np.std(s)

print('Responder 비율:', y.mean().round(2), '| n =', len(y))
for name, make, X in [('Clinical-only', clin_pipe, Xc),
                      ('EEG-only (CBraMod frozen)', eeg_pipe, Xe),
                      ('Fusion (concat)', fuse_pipe, Xf)]:
    m_, s_ = cv_auc(make, X)
    print(f'{name:28s} AUROC {m_:.3f} ± {s_:.3f}')

## 8. EEG-only 라벨 순열 검정 (100회)

In [ ]:
rng = np.random.default_rng(0)
real, _ = cv_auc(eeg_pipe, Xe, seeds=3)
null = []
y_orig = y.copy()
for i in range(100):
    y = rng.permutation(y_orig)
    null.append(cv_auc(eeg_pipe, Xe, seeds=1)[0])
y = y_orig
p = (np.sum(np.array(null) >= real) + 1) / (len(null) + 1)
print(f'EEG-only AUROC {real:.3f} | 무작위 평균 {np.mean(null):.3f} | p = {p:.3f}')

## 9. 결합 방식 비교 (early · late 평균 · stacking)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from scipy.special import logit

y = y_orig.copy()

def fit_pred(make, Xtr, ytr, Xte):
    m = make().fit(Xtr, ytr)
    return m.predict_proba(Xte)[:, 1]

def early(tr, te):
    return fit_pred(fuse_pipe, Xf[tr], y[tr], Xf[te])

def late_avg(tr, te):
    pc = fit_pred(clin_pipe, Xc[tr], y[tr], Xc[te])
    pe = fit_pred(eeg_pipe,  Xe[tr], y[tr], Xe[te])
    return (pc + pe) / 2

meta_weights = []
def stacking(tr, te):
    inner = StratifiedKFold(5, shuffle=True, random_state=0)
    oc = cross_val_predict(clin_pipe(), Xc[tr], y[tr], cv=inner, method='predict_proba')[:, 1]
    oe = cross_val_predict(eeg_pipe(),  Xe[tr], y[tr], cv=inner, method='predict_proba')[:, 1]
    Z = logit(np.clip(np.c_[oc, oe], 1e-4, 1 - 1e-4))
    meta = LogisticRegression(max_iter=5000).fit(Z, y[tr])
    meta_weights.append(meta.coef_[0])
    pc = fit_pred(clin_pipe, Xc[tr], y[tr], Xc[te])
    pe = fit_pred(eeg_pipe,  Xe[tr], y[tr], Xe[te])
    Zte = logit(np.clip(np.c_[pc, pe], 1e-4, 1 - 1e-4))
    return meta.predict_proba(Zte)[:, 1]

def clin_only(tr, te): return fit_pred(clin_pipe, Xc[tr], y[tr], Xc[te])
def eeg_only(tr, te):  return fit_pred(eeg_pipe,  Xe[tr], y[tr], Xe[te])

methods = {'Clinical-only': clin_only, 'EEG-only': eeg_only,
           'Early fusion (concat)': early, 'Late fusion (avg)': late_avg,
           'Late fusion (stacking)': stacking}

SEEDS = 10
res = {k: [] for k in methods}
for seed in range(SEEDS):
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    splits = list(cv.split(Xc, y))
    for name, fn in methods.items():
        p = np.zeros(len(y))
        for tr, te in splits:
            p[te] = fn(tr, te)
        res[name].append(roc_auc_score(y, p))

base = np.array(res['Clinical-only'])
print(f"{'method':26s} AUROC          vs Clinical-only (차이, 이긴 비율)")
for name, s in res.items():
    s = np.array(s); diff = s - base
    print(f"{name:26s} {s.mean():.3f} ± {s.std():.3f}   {diff.mean():+.3f}, {np.mean(diff > 0):.0%}")

w = np.array(meta_weights)
print('\nstacking 가중치 평균  임상: %.2f | EEG: %.2f' % tuple(w.mean(axis=0)))

## 10. 무작위 가중치 CBraMod 대조
사전학습이 실제로 의미 있는 특징을 주는지 확인합니다. (1) 눈 뜸 vs 감음 구분 (2) Responder 예측을 사전학습 / 무작위 가중치로 비교

In [ ]:
import copy
from sklearn.model_selection import GroupKFold

def emb_for(cond, encoder):
    global model
    old = model; model = encoder
    out = []
    for sid in ok_ids:
        ses = int(mdd.set_index('TDBRAIN_ID').loc[sid, 'sessID'])
        f = list(EEG_ROOT.glob(f'{sid}/ses-{ses}/eeg/*task-{cond}_eeg.bdf'))[0]
        out.append(subject_embedding(f)[0])
    model = old
    return np.stack(out)

# 무작위 가중치 모델
rand_model = copy.deepcopy(model)
for m in rand_model.modules():
    if hasattr(m, 'reset_parameters'):
        m.reset_parameters()
rand_model.eval()

def eo_ec_auc(E_ec, E_eo):
    X = np.vstack([E_ec, E_eo])
    yy = np.r_[np.zeros(len(E_ec)), np.ones(len(E_eo))]
    groups = np.r_[np.arange(len(E_ec)), np.arange(len(E_eo))]   # 같은 사람은 같은 fold
    p = cross_val_predict(eeg_pipe(), X, yy, cv=GroupKFold(5), groups=groups, method='predict_proba')[:, 1]
    return roc_auc_score(yy, p)

E_ec_pre = emb                                   # 이미 뽑은 사전학습 EC 임베딩
E_eo_pre = emb_for('restEO', model)
E_ec_rnd = emb_for('restEC', rand_model)
E_eo_rnd = emb_for('restEO', rand_model)

print('눈 뜸 vs 감음 | 사전학습:', round(eo_ec_auc(E_ec_pre, E_eo_pre), 3),
      '| 무작위:', round(eo_ec_auc(E_ec_rnd, E_eo_rnd), 3))

In [ ]:
y = y_orig.copy()
print('Responder | 사전학습 EEG-only:', round(cv_auc(eeg_pipe, E_ec_pre)[0], 3),
      '| 무작위 EEG-only:', round(cv_auc(eeg_pipe, E_ec_rnd)[0], 3))